In [3]:
import numpy as np
import os
from scipy.linalg import orthogonal_procrustes

embedding_dir = '../Datasets/decade_svd_300'
vocab_dir = '../Datasets/decade_stats'
output_dir = '../Datasets/decade_aligned'
os.makedirs(output_dir, exist_ok=True)

def pad_vectors(vecs, target_dim=300):
    """Ensures the matrix has exactly 300 columns, filling with zeros if needed."""
    if vecs.shape[1] < target_dim:
        padding = np.zeros((vecs.shape[0], target_dim - vecs.shape[1]))
        return np.hstack((vecs, padding))
    return vecs[:, :target_dim]

def align_pair(base_vecs, other_vecs, base_vocab, other_vocab):
    common_vocab = [w for w in base_vocab if w in other_vocab]
    base_idx = {w: i for i, w in enumerate(base_vocab)}
    other_idx = {w: i for i, w in enumerate(other_vocab)}
    
    indices_base = [base_idx[w] for w in common_vocab]
    indices_other = [other_idx[w] for w in common_vocab]
    
    # Extract shared vectors and force 300 dim
    A = pad_vectors(base_vecs[indices_base], 300)
    B = pad_vectors(other_vecs[indices_other], 300)
    
    # A.shape will be (N, 300) and B.shape will be (N, 300)
    R, _ = orthogonal_procrustes(A, B)
    
    full_other_padded = pad_vectors(other_vecs, 300)
    return full_other_padded.dot(R)

# Alignment
all_files = sorted([f for f in os.listdir(embedding_dir) if f.endswith("-u.npy")])
base_vecs = None
base_vocab = None

for filename in all_files:
    decade = filename.split("-u.npy")[0]
    print(f"Aligning decade: {decade}")
    
    curr_vecs = np.load(os.path.join(embedding_dir, filename))
    with open(os.path.join(vocab_dir, f"{decade}_vocab.txt"), "r", encoding='utf-8') as f:
        curr_vocab = [line.strip() for line in f]
        
    if base_vecs is None:
        # Pad the anchor decade to 300 as well
        aligned_vecs = pad_vectors(curr_vecs, 300)
    else:
        aligned_vecs = align_pair(base_vecs, curr_vecs, base_vocab, curr_vocab)
        
    np.save(os.path.join(output_dir, f"{decade}-aligned.npy"), aligned_vecs)
    base_vecs = aligned_vecs
    base_vocab = curr_vocab

print("Alignment successful with shape correction!")

Aligning decade: 1810s
Aligning decade: 1820s
Aligning decade: 1830s
Aligning decade: 1840s
Aligning decade: 1850s
Aligning decade: 1860s
Aligning decade: 1870s
Aligning decade: 1880s
Aligning decade: 1890s
Aligning decade: 1900s
Aligning decade: 1910s
Aligning decade: 1920s
Aligning decade: 1930s
Aligning decade: 1940s
Aligning decade: 1950s
Aligning decade: 1960s
Aligning decade: 1970s
Aligning decade: 1980s
Aligning decade: 1990s
Alignment successful with shape correction!
